In [ ]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import search_around_sky



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def populate_sdss_fields(df):

    hdul = fits.open('data/dr16q_prop_May01_2024.fits')
    fits_data = hdul[1].data  # Assuming the data is in the first extension    
    fits_data2 = hdul[2].data  # Assuming the data is in the second extension

    
    # Add apparent_mag_i as a new field to fits_data2

    # Convert fits_data2 to Table, add column, convert back to FITS_rec
    table_data2 = Table(fits_data2)

    # Calculate apparent_mag_i from PSFFLUX (i-band is index 2)
    apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5
    table_data2['apparent_mag_i'] = apparent_mag_i

    fits_data2 = table_data2.as_array()

    fields1 = ['RA', 'DEC', 'SDSS_NAME', 'Z_SYS', 'LOGLBOL', 'LOGLBOL_ERR', 'LOGL3000', 'LOGL3000_ERR']
    fields2 = ['M_I', 'SN_MEDIAN_ALL', 'apparent_mag_i']

    data = {}
    for f in fields1:
        data[f] = fits_data[f]
    for f in fields2:
        data[f] = fits_data2[f]

    df_fits = pd.DataFrame(data)
    df_fits['Z'] = df_fits['Z_SYS']

    # Calculate LOGLBOL for each row in df_fits based on Z_SYS
    df_fits['LOGLBOL_CALC'] = np.where(
        df_fits['Z'] < 0.7,
        np.log10(5.15) + df_fits['LOGL3000'],
        df_fits['LOGLBOL']
    )
    df_fits['LOGLBOL_ERR_CALC'] = np.where(
        df_fits['Z'] < 0.7,
        df_fits['LOGL3000_ERR'],
        df_fits['LOGLBOL_ERR']
    )

    df_fits['LOGLBOL'] = df_fits['LOGLBOL_CALC']
    df_fits['LOGLBOL_ERR'] = df_fits['LOGLBOL_ERR_CALC']

    cat = pd.read_parquet(f"data/S82/Catalog.parquet").set_index('idx')

    df_cat = cat.reset_index()[['objectId', 'RA', 'DEC']]
    df_cat = df_cat.rename(columns={'RA': 'RA_w', 'DEC': 'DEC_w'})

    # SkyCoord for both catalogs (assumes RA/DEC are in degrees)
    coords_cat  = SkyCoord(ra=df_cat['RA_w'].values * u.deg, dec=df_cat['DEC_w'].values * u.deg)
    coords_fits = SkyCoord(ra=df_fits['RA'].values    * u.deg, dec=df_fits['DEC'].values    * u.deg)

    # All matches within 1 arcsec
    idx_cat, idx_fits, sep2d, _ = search_around_sky(coords_cat, coords_fits, 1 * u.arcsec)

    # Give df_fits a key to merge on
    df_fits = df_fits.copy()
    df_fits['idx'] = np.arange(len(df_fits))

    # Take only the matched rows from df_cat (use iloc!), attach the matching df_fits index
    df_cat_match = df_cat.iloc[idx_cat].reset_index(drop=True).copy()
    df_cat_match['idx'] = idx_fits
    df_cat_match['sep_arcsec'] = sep2d.to(u.arcsec).value  # handy to keep

    # Inner join: repeats df_fits rows if there are multiple cat matches (expected behavior)
    df_catalog_merged = pd.merge(df_fits, df_cat_match, on='idx', how='inner')

    df_merged_final = pd.merge(df, df_catalog_merged, left_on='object_id', right_on='objectId', how='inner')
    df_merged_final = df_merged_final[df_merged_final['LOGLBOL'] > 43]

    df_merged_final['ra'] = df_merged_final['RA_w']
    df_merged_final['dec'] = df_merged_final['DEC_w']

    return df_merged_final


In [10]:
df = pd.read_csv('data/sample_stone_fittedm2500.csv', dtype={'object_id': str})

df_populated = populate_sdss_fields(df)
df_populated.keys()

df_populated.to_csv('data/sample_stone_fittedm2500.csv', index=False)


/tmp/ipykernel_3233351/3580419258.py:8: RuntimeWarning: divide by zero encountered in log10
  apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5
/tmp/ipykernel_3233351/3580419258.py:8: RuntimeWarning: invalid value encountered in log10
  apparent_mag_i = -2.5 * np.log10(fits_data2['PSFFLUX'][:, 2]) + 22.5
